In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA = Path("../data/raw")  # use Path("data/raw") if running from project root

mips = pd.read_csv(RAW_DATA / "grp_public_reporting.csv", dtype=str)
dac = pd.read_csv(RAW_DATA / "DAC_NationalDownloadableFile.csv", dtype=str)

# Clean column names because MIPS has leading spaces
mips.columns = mips.columns.str.strip()
dac.columns = dac.columns.str.strip()

# Make join keys consistent
mips["org_pac_id"] = mips["org_PAC_ID"].astype(str).str.strip()
dac["org_pac_id"] = dac["org_pac_id"].astype(str).str.strip()

In [15]:
zip_county = pd.read_csv(RAW_DATA / "ZIP-COUNTY.csv", dtype=str)

# Clean column names because MIPS has leading spaces
zip_county.columns = zip_county.columns.str.strip()

zip_county

,zip,geoid,res_ratio,bus_ratio,oth_ratio,tot_ratio,city,state
0,00501,36103,0.000000000,1.000000000,0.000000000,1.000000000,HOLTSVILLE,NY
1,00601,72001,0.997451580,0.994936709,0.987951807,0.997052893,ADJUNTAS,PR
2,00601,72081,0.002548420,0.005063291,0.012048193,0.002947107,ADJUNTAS,PR
3,00602,72003,0.999416278,0.998983740,1.000000000,0.999394856,AGUADA,PR
4,00602,72117,0.000583722,0.000000000,0.000000000,0.000529501,AGUADA,PR
...,...,...,...,...,...,...,...,...
54557,99926,02198,1.000000000,0.000000000,1.000000000,1.000000000,METLAKATLA,AK
54558,99927,02198,0.000000000,0.000000000,1.000000000,1.000000000,POINT BAKER,AK
54559,99928,02130,0.000000000,0.000000000,1.000000000,1.000000000,WARD COVE,AK
54560,99929,02275,1.000000000,1.000000000,1.000000000,1.000000000,WRANGELL,AK


In [17]:
len(zip_county['zip']), len(zip_county['geoid'])

(54562, 54562)

KeyError: 'county_fips'

In [2]:
def clean_zip5(x):
    if pd.isna(x):
        return pd.NA
    
    x = str(x).strip()
    
    # Remove decimal artifact like "602.0"
    if x.endswith(".0"):
        x = x[:-2]
    
    # Keep only digits
    digits = "".join(ch for ch in x if ch.isdigit())
    
    if len(digits) == 0:
        return pd.NA
    
    # If ZIP+4 or longer, keep first 5
    # If short like Puerto Rico 602, pad to 00602
    return digits[:5].zfill(5)

dac["zip5"] = dac["ZIP Code"].apply(clean_zip5)

dac[["org_pac_id", "ZIP Code", "zip5"]].head()

,org_pac_id,ZIP Code,zip5
0,nan,602,00602
1,nan,602,00602
2,nan,622,00622
3,nan,646,00646
4,6305731118,646,00646


In [3]:
# Clean org_pac_id column in dac, remove na values (individual physicians have no org_pac_id)

dac["org_pac_id"] = dac["org_pac_id"].astype("string").str.strip()

dac_group = dac[
    dac["org_pac_id"].notna() &
    dac["org_pac_id"].ne("") &
    dac["org_pac_id"].str.lower().ne("nan")
].copy()

dac_group[["org_pac_id", "ZIP Code", "zip5"]].head()

,org_pac_id,ZIP Code,zip5
4,6305731118,646,00646
17,9638151756,802,00802
18,2860304482,840,00840
19,4385740141,840,00840
20,2264424712,907,00907


In [ ]:
#merge mips and dac_group on org_pac_id
merged = pd.merge(mips, dac_group, on="org_pac_id", how="left", suffixes=("_mips", "_dac"))


np.int64(97497)

In [10]:
missing_zip_pct=merged['zip5'].isna().sum()/len(merged)*100
print(f"{missing_zip_pct:.2f}% of rows are missing zip5")

0.90% of rows are missing zip5


In [11]:
merged_no_missing_zip = merged.dropna(subset=["zip5"])

In [19]:


# ZIP-to-county crosswalk
# Standardize join keys
zip_county["zip5"] = zip_county["zip"].astype(str).str.zfill(5)
zip_county["county_fips"] = zip_county["geoid"].astype(str).str.zfill(5)

merged_no_missing_zip["zip5"] = merged_no_missing_zip["zip5"].astype(str).str.zfill(5)

# Merge MIPS/DAC ZIP data to county crosswalk
mips_zip_county = merged_no_missing_zip.merge(
    zip_county[["zip5", "county_fips", "tot_ratio", "res_ratio", "city", "state"]],
    on="zip5",
    how="left"
)

# Check ZIP-to-county match rate
print("Missing county_fips after ZIP-county merge:")
print(mips_zip_county["county_fips"].isna().mean() * 100)

C:\Users\zslet\AppData\Local\Temp\ipykernel_6968\278393885.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_no_missing_zip["zip5"] = merged_no_missing_zip["zip5"].astype(str).str.zfill(5)


Missing county_fips after ZIP-county merge:
3.022008246946399


In [20]:

geo = pd.read_csv(
    RAW_DATA / "2014-2024 Original Medicare Geographic Variation Public Use File.csv",
    dtype=str
)

geo.columns = geo.columns.str.strip()

# Standardize county FIPS in geo
geo["county_fips"] = geo["BENE_GEO_CD"].astype(str).str.zfill(5)

print(geo["BENE_GEO_LVL"].value_counts(dropna=False))
print(geo["BENE_AGE_LVL"].value_counts(dropna=False))
print(geo["YEAR"].value_counts(dropna=False).sort_index())

BENE_GEO_LVL
County      35146
State        1815
National       33
Name: count, dtype: int64
BENE_AGE_LVL
All     35762
<65       616
>=65      616
Name: count, dtype: int64
YEAR
2014    3362
2015    3363
2016    3362
2017    3362
2018    3362
2019    3362
2020    3362
2021    3364
2022    3365
2023    3365
2024    3365
Name: count, dtype: int64


In [21]:
geo_county = geo[
    geo["BENE_GEO_LVL"].str.lower().eq("county")
].copy()